# Figure 9: [Latent] stacked valid/invalid unique names per model, round 100

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.plots.style import apply_paper_style, model_label, model_color, ordered_models
apply_paper_style()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.data_loader import load_latent_professions, try_load, warn_incomplete_coverage

latent = try_load(load_latent_professions, expanded=True, label="latent professions (expanded)")
if latent is not None:
    warn_incomplete_coverage(latent, label="latent")


In [ ]:
if latent is None:
    print("Skipping Fig 9: no latent data loaded.")
else:
    # Unique canonical names per model, split by validity (matches the "excluded_people_after_round"
    # set at the final round: the expanded loader already gives one row per unique name per query).
    valid_counts = latent[latent["validity_label"] == True].groupby("model_version")["resolved_name"].apply(lambda s: len(set(s)))
    invalid_counts = latent[latent["validity_label"] != True].groupby("model_version")["resolved_name"].apply(lambda s: len(set(s)))
    models = valid_counts.add(invalid_counts, fill_value=0).sort_values(ascending=False).index.tolist()

    if not models:
        print("No models with any latent data: nothing to plot.")
    else:
        fig, ax = plt.subplots(figsize=(7, 5))
        valid_vals = [valid_counts.get(m, 0) for m in models]
        invalid_vals = [invalid_counts.get(m, 0) for m in models]
        ax.bar([model_label(m) for m in models], valid_vals, label="Valid", color="#4C78A8")
        ax.bar([model_label(m) for m in models], invalid_vals, bottom=valid_vals, label="Invalid", color="#d62728")
        ax.set_ylabel("# Unique Named People")
        ax.legend()
        plt.xticks(rotation=60, ha="right")
        plt.tight_layout()
        plt.show()
